# Vertex AI vision-only resume field evaluation

Runs every annotated resume through one configurable Vertex AI model, saves every attempt to an append-only JSONL ledger, and computes field-level confusion counts. Descriptions are deliberately excluded.

**Metric semantics:** item-level TP/FP/FN compare extracted values with gold values. An item-level TN is undefined because the universe of possible skills, employers, degrees, etc. is unbounded. Therefore TN and specificity are reported only for the separate, well-defined question: *is this field present on this resume?*

The notebook is restart-safe: successful `(resume, model, prompt)` evaluations are skipped, failures remain in the audit ledger, and graphs can be rebuilt without calling Vertex AI again.

## 1. Setup

Run this notebook with the backend environment (`uv sync`, then select that kernel). Authenticate locally with `gcloud auth application-default login`, or use a service account in your Vertex environment.

In [ ]:
from __future__ import annotations

import asyncio
import hashlib
import importlib
import json
import os
import random
import sys
import time
import uuid
from dataclasses import replace
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from jsonschema import Draft7Validator

def find_backend(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / 'pyproject.toml').exists() and (candidate / 'src').exists():
            return candidate
    nested = start / 'Group-4-DS-and-AI-Lab-Project' / 'backend'
    if nested.exists():
        return nested
    raise RuntimeError('Could not locate the backend directory.')

BACKEND = find_backend(Path.cwd().resolve())
if str(BACKEND) not in sys.path:
    sys.path.insert(0, str(BACKEND))

from evals import gold
from src.core.config import get_settings
from src.resume_parsing.internal.location import locality_only
from src.resume_parsing.internal.pipeline import extraction, postprocess, preprocess, routing
from src.resume_parsing.internal.prompts import SYSTEM_PROMPT, load_extraction_schema, load_schema
from src.resume_parsing.internal.providers import vertex_ai
vertex_ai = importlib.reload(vertex_ai)  # Never retain a stale provider in Jupyter.
build_vertex_provider = vertex_ai.build_vertex_provider
from src.resume_parsing.schemas import ParseRoute

print('Backend:', BACKEND)

In [ ]:
# Edit these values before running model calls. Do not put credentials in the notebook.
PROJECT_ID = 'project-ad1a812b-a085-4bcc-ad0'
LOCATION = 'global'
MODEL_NAME = 'gemma-4-26b-a4b-it-maas'
# Experiment-controlled values: update these together when testing a new prompt.
PROMPT_VERSION = 'v016_gemma_v010_comparable'
PROMPT_SNAPSHOT_PATH = BACKEND / 'experiments' / 'prompts' / 'vertex_v010_system_prompt.txt'
PROMPT_TEXT = PROMPT_SNAPSHOT_PATH.read_text(encoding='utf-8').rstrip('\n')
assert PROMPT_TEXT.strip(), 'The versioned prompt snapshot is empty.'
TEMPERATURE = 0.0
THINKING_LEVEL = 'medium'  # Recorded for a stable experiment identity; Gemma ignores Gemini thinking controls
MAX_RETRIES = 4       # five total attempts for transient Vertex capacity errors
BASE_RETRY_SECONDS = 30.0  # exponential waits: ~30s, 60s, 120s, 240s
REQUEST_TIMEOUT_SECONDS = 300.0  # allow long/scanned resumes to finish
RENDER_DPI = get_settings().resume_render_dpi  # production preprocessing setting
INPUT_USD_PER_MILLION_TOKENS = 0.15
OUTPUT_USD_PER_MILLION_TOKENS = 0.60
CACHE_HIT_USD_PER_MILLION_TOKENS = 0.015
PRICING_VERSION = 'gemma_4_26b_vertex_maas_2026-08-09'
SELECTED_RESUME_IDS = None  # all annotated resumes
LIMIT = None                # all 86; five completed smoke rows are restart-safely skipped
SPLIT = None              # dev + test: all 86 annotated resumes
FORCE_RERUN = False
NORMALIZE_FOR_SCORING = False  # keep inference evidence raw; analysis derives normalized scoring
LOCATION_POLICY_VERSION = 'locality_only_v001'

EVALUATION_MODE = 'gemma4_26b_same_prompt_as_best_gemini_v010_v016'
RUNS_DIR = BACKEND / 'experiments' / 'runs' / 'vertex_vision_raw_baseline'
LEDGER_PATH = RUNS_DIR / 'attempts.jsonl'
EXPORT_DIR = RUNS_DIR / 'exports'
RUNS_DIR.mkdir(parents=True, exist_ok=True)
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

assert PROJECT_ID != 'YOUR_GCP_PROJECT_ID', 'Set GOOGLE_CLOUD_PROJECT or edit PROJECT_ID.'
print({'project': PROJECT_ID, 'location': LOCATION, 'model': MODEL_NAME, 'ledger': str(LEDGER_PATH)})

## 2. Load and audit the gold dataset

The count is discovered rather than hard-coded. The cleaned source ledger contains 86 annotated resumes. Uniqueness and file existence are asserted before any paid model calls.

In [ ]:
examples = gold.load(SPLIT)
if SELECTED_RESUME_IDS is not None:
    selected_ids = set(SELECTED_RESUME_IDS)
    examples = [e for e in examples if e.resume_id in selected_ids]
    missing_ids = selected_ids - {e.resume_id for e in examples}
    assert not missing_ids, f'Unknown selected resume IDs: {sorted(missing_ids)}'
if LIMIT is not None:
    examples = examples[:LIMIT]

dataset_audit = pd.DataFrame({
    'resume_id': [e.resume_id for e in examples],
    'split': [e.split for e in examples],
    'category': [e.category for e in examples],
    'file_exists': [e.pdf_path.exists() for e in examples],
})
assert dataset_audit['resume_id'].is_unique, 'Duplicate resume IDs found in gold.jsonl.'
assert dataset_audit['file_exists'].all(), 'One or more source files are missing.'
display(dataset_audit.groupby('split').size().rename('resumes').to_frame())
display(dataset_audit.groupby('category').size().sort_values(ascending=False).rename('resumes').to_frame())
print('Selected resumes:', len(examples))

## 3. Field definitions and confusion counts

Repeated nested fields are flattened across entries. This baseline compares values exactly as returned and applies no value transformations.

In [ ]:
FIELD_SPECS = {
    'contact.name': ('scalar', 'contact', 'name'),
    'contact.location': ('scalar', 'contact', 'location'),
    'skills': ('list', 'skills'),
    'education.degree': ('nested', 'education', 'degree'),
    'education.field': ('nested', 'education', 'field'),
    'education.institution': ('nested', 'education', 'institution'),
    'education.start_year': ('nested', 'education', 'start_year'),
    'education.end_year': ('nested', 'education', 'end_year'),
    'experience.job_title': ('nested', 'experience', 'job_title'),
    'experience.company': ('nested', 'experience', 'company'),
    'experience.location': ('nested', 'experience', 'location'),
    'experience.start_date': ('nested', 'experience', 'start_date'),
    'experience.end_date': ('nested', 'experience', 'end_date'),
    'experience.current_role': ('nested', 'experience', 'current_role'),
    'projects.name': ('nested', 'projects', 'name'),
    'projects.technologies': ('nested', 'projects', 'technologies'),
    'certifications.name': ('nested', 'certifications', 'name'),
    'certifications.issuer': ('nested', 'certifications', 'issuer'),
    'certifications.year': ('nested', 'certifications', 'year'),
}

def comparison_value(value: Any, field: str) -> str:
    """Represent a value for scoring; exact strings are the baseline."""
    if value is None or value == '':
        return ''
    if isinstance(value, str):
        if NORMALIZE_FOR_SCORING and field.endswith('.location'):
            value = locality_only(value) or ''
        return ' '.join(value.casefold().split()) if NORMALIZE_FOR_SCORING else value
    return json.dumps(value, ensure_ascii=False, sort_keys=True)

def values_for(profile: dict, spec: tuple[str, ...], field: str) -> set[str]:
    kind, *path = spec
    if kind == 'scalar':
        value = (profile.get(path[0]) or {}).get(path[1])
        comparable = comparison_value(value, field)
        return {comparable} if comparable else set()
    if kind == 'list':
        node: Any = profile
        for key in path:
            node = (node or {}).get(key) if isinstance(node, dict) else []
        return {comparison_value(v, field) for v in (node or []) if comparison_value(v, field)}
    section, key = path
    values: set[str] = set()
    for entry in profile.get(section) or []:
        if not isinstance(entry, dict):
            continue
        value = entry.get(key)
        raw_values = value if isinstance(value, list) else [value]
        values |= {comparison_value(v, field) for v in raw_values if comparison_value(v, field)}
    return values

def confusion_rows(prediction: dict, reference: dict) -> list[dict]:
    rows = []
    for field, spec in FIELD_SPECS.items():
        predicted = values_for(prediction, spec, field)
        expected = values_for(reference, spec, field)
        tp_values = sorted(predicted & expected)
        fp_values = sorted(predicted - expected)
        fn_values = sorted(expected - predicted)
        rows.append({
            'field': field,
            'tp': len(tp_values),
            'fp': len(fp_values),
            'fn': len(fn_values),
            'predicted_values': sorted(predicted),
            'gold_values': sorted(expected),
            'tp_values': tp_values,
            'fp_values': fp_values,
            'fn_values': fn_values,
            # Presence/absence classification has a finite negative class.
            'presence_tp': int(bool(predicted) and bool(expected)),
            'presence_fp': int(bool(predicted) and not expected),
            'presence_fn': int(not predicted and bool(expected)),
            'presence_tn': int(not predicted and not expected),
            'predicted_count': len(predicted),
            'gold_count': len(expected),
        })
    return rows

# Sanity check: exact self-comparison must produce no FP or FN.
probe = examples[0].profile
assert all(r['fp'] == 0 and r['fn'] == 0 for r in confusion_rows(probe, probe))

## 4. Vertex AI provider and append-only ledger

This class uses the Google Gen AI SDK in Vertex mode. Each PDF page is rendered to an image, so this is the vision path only. Gemma receives the schema in its user prompt; Gemini-family models additionally use structured output.

In [ ]:
class CapturingProvider:
    """Capture schema evidence with exact-address location data removed."""
    def __init__(self, delegate):
        self.delegate = delegate
        self.raw_page_outputs = []

    async def extract(self, page, *, model: str) -> dict:
        payload = await self.delegate.extract(page, model=model)
        safe_payload = json.loads(json.dumps(payload))
        contact = safe_payload.get('contact') or {}
        if isinstance(contact, dict):
            contact['location'] = locality_only(contact.get('location'))
        for entry in safe_payload.get('experience') or []:
            if isinstance(entry, dict):
                entry['location'] = locality_only(entry.get('location'))
        self.raw_page_outputs.append({'page_index': page.index, 'payload': safe_payload})
        return payload

def drain_usage_summary(provider) -> dict:
    events = provider.drain_usage() if hasattr(provider, 'drain_usage') else []
    prompt_tokens = sum(int(row.get('prompt_tokens') or 0) for row in events)
    output_tokens = sum(int(row.get('output_tokens') or 0) for row in events)
    thinking_tokens = sum(int(row.get('thinking_tokens') or 0) for row in events)
    total_tokens = sum(int(row.get('total_tokens') or 0) for row in events)
    cached_tokens = sum(int(row.get('cached_input_tokens') or 0) for row in events)
    uncached_tokens = max(prompt_tokens - cached_tokens, 0)
    estimated_cost = (
        uncached_tokens * INPUT_USD_PER_MILLION_TOKENS
        + cached_tokens * CACHE_HIT_USD_PER_MILLION_TOKENS
        + (output_tokens + thinking_tokens) * OUTPUT_USD_PER_MILLION_TOKENS
    ) / 1_000_000
    return {
        'usage_metadata_available': bool(events),
        'prompt_tokens': prompt_tokens if events else None,
        'output_tokens': output_tokens if events else None,
        'thinking_tokens': thinking_tokens if events else None,
        'total_tokens': total_tokens if events else None,
        'cached_input_tokens': cached_tokens if events else None,
        'estimated_cost_usd': round(estimated_cost, 8) if events else None,
        'pricing_version': PRICING_VERSION,
        'provider_usage_events': events,
    }

async def extract_resume(provider, pdf_path: Path, model: str) -> tuple[dict, dict, dict]:
    drain_usage_summary(provider)  # discard stale usage before this attempt
    resume_started = time.perf_counter()
    content = pdf_path.read_bytes()
    route_started = time.perf_counter()
    document = routing.route(pdf_path.name, 'application/pdf', content)
    route_seconds = time.perf_counter() - route_started
    # This experiment changes only the chosen input route; every pipeline
    # stage after that is the production implementation.
    document = replace(document, route=ParseRoute.VISION)
    preprocess_started = time.perf_counter()
    pages = preprocess.to_pages(document)
    preprocess_seconds = time.perf_counter() - preprocess_started
    inference_started = time.perf_counter()
    capturing_provider = CapturingProvider(provider)
    page_profiles = await asyncio.wait_for(
        extraction.extract_pages(capturing_provider, pages, model=model),
        timeout=REQUEST_TIMEOUT_SECONDS,
    )
    usage = drain_usage_summary(provider)
    model_wall_seconds = time.perf_counter() - inference_started
    timings = {
        'page_count': len(pages),
        'route_seconds': round(route_seconds, 3),
        'preprocess_seconds': round(preprocess_seconds, 3),
        'model_wall_seconds': round(model_wall_seconds, 3),
        'input_image_bytes': sum(len(page.image_png or b'') for page in pages),
        'resume_total_seconds': round(time.perf_counter() - resume_started, 3),
        **usage,
    }
    raw_pages = sorted(capturing_provider.raw_page_outputs, key=lambda row: row['page_index'])
    if len(raw_pages) != 1:
        raise RuntimeError(
            'Raw strict baseline currently requires one-page resumes; '
            f'found {len(raw_pages)} successful pages.'
        )
    raw_prediction = raw_pages[0]['payload']
    production_prediction = postprocess.merge(page_profiles).model_dump(mode='json')
    return raw_prediction, production_prediction, timings

def raw_schema_errors(payload: dict) -> list[str]:
    validator = Draft7Validator(load_extraction_schema())
    errors = []
    for error in sorted(validator.iter_errors(payload), key=str):
        location = '.'.join(str(part) for part in error.absolute_path) or '<root>'
        errors.append(f'{location}: {error.message}')
    return errors

def prompt_hash() -> str:
    material = PROMPT_TEXT + json.dumps(load_extraction_schema(), sort_keys=True)
    return hashlib.sha256(material.encode()).hexdigest()

def append_jsonl(path: Path, row: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    payload = json.dumps(row, ensure_ascii=False, sort_keys=True) + '\n'
    with path.open('a', encoding='utf-8') as stream:
        stream.write(payload)
        stream.flush()
        os.fsync(stream.fileno())

def read_ledger(path: Path = LEDGER_PATH) -> list[dict]:
    if not path.exists():
        return []
    rows = []
    for line_number, line in enumerate(path.read_text(encoding='utf-8').splitlines(), 1):
        if not line.strip():
            continue
        try:
            rows.append(json.loads(line))
        except json.JSONDecodeError as exc:
            raise ValueError(f'Corrupt ledger line {line_number}: {exc}') from exc
    return rows

PROMPT_SHA256 = prompt_hash()
PROVIDER_SHA256 = hashlib.sha256(Path(vertex_ai.__file__).read_bytes()).hexdigest()
EXPERIMENT_KEY = (
    f'{MODEL_NAME}:{PROMPT_VERSION}:{PROMPT_SHA256}:'
    f'temp={TEMPERATURE}:thinking={THINKING_LEVEL}:dpi={RENDER_DPI}:evaluation={EVALUATION_MODE}:'
    f'normalize_for_scoring={NORMALIZE_FOR_SCORING}:'
    f'location_policy={LOCATION_POLICY_VERSION}:'
    f'provider={PROVIDER_SHA256}:gold={gold.GOLD_CORRECTIONS_VERSION}'
)
print('Experiment key:', EXPERIMENT_KEY[:80])

## 5. Run or resume the evaluation

This experiment runs Gemma 4 26B on all 86 annotated resumes using the exact v10 prompt snapshot from the best complete Gemini run. This isolates model choice for the quality, latency, token-usage and estimated-cost comparison. Successful rows are restart-safely skipped and every attempt remains append-only. Raw predictions and gold values contain retained personal data, so the ledger lives under the git-ignored `experiments/runs/` directory.

In [ ]:
provider = build_vertex_provider(
    project=PROJECT_ID, location=LOCATION, temperature=TEMPERATURE,
    thinking_level=THINKING_LEVEL,
    timeout_seconds=REQUEST_TIMEOUT_SECONDS,
    system_prompt=PROMPT_TEXT,
)
run_id = str(uuid.uuid4())
prior_rows = read_ledger()
completed = {
    row['resume_id'] for row in prior_rows
    if row.get('status') == 'success' and row.get('experiment_key') == EXPERIMENT_KEY
}
pending = [
    (index, example) for index, example in enumerate(examples, start=1)
    if FORCE_RERUN or example.resume_id not in completed
]
print(f'{len(completed)} already successful; {len(pending)} pending in this selection.')

async def run_all() -> None:
    total = len(examples)
    for resume_index, example in tqdm(pending, desc='Resumes'):
        print(
            f'[{resume_index}/{total}] {example.resume_id}: starting',
            flush=True,
        )
        started = time.perf_counter()
        for attempt in range(MAX_RETRIES + 1):
            base = {
                'record_id': str(uuid.uuid4()),
                'run_id': run_id,
                'timestamp_utc': datetime.now(timezone.utc).isoformat(),
                'experiment_key': EXPERIMENT_KEY,
                'resume_id': example.resume_id,
                'resume_sha256': hashlib.sha256(example.pdf_path.read_bytes()).hexdigest(),
                'split': example.split,
                'category': example.category,
                'model': MODEL_NAME,
                'project': PROJECT_ID,
                'location': LOCATION,
                'prompt_version': PROMPT_VERSION,
                'prompt_sha256': PROMPT_SHA256,
                'provider_sha256': PROVIDER_SHA256,
                'system_prompt': PROMPT_TEXT,
                'response_schema': load_extraction_schema(),
                'temperature': TEMPERATURE,
                'render_dpi': RENDER_DPI,
                'pipeline': 'production_vision',
                'evaluation_mode': EVALUATION_MODE,
                'scored_output': 'production_prediction',
                'gold_transform': f'schema_key_mapping_locality_and_{gold.GOLD_CORRECTIONS_VERSION}',
                'location_policy_version': LOCATION_POLICY_VERSION,
                'normalize_for_scoring': NORMALIZE_FOR_SCORING,
                'value_normalization': 'case_whitespace' if NORMALIZE_FOR_SCORING else 'none',
                'stored_location_policy': 'locality_only_no_exact_address',
                'attempt': attempt + 1,
            }
            try:
                raw_prediction, production_prediction, timings = await extract_resume(
                    provider, example.pdf_path, MODEL_NAME
                )
                failure_usage = drain_usage_summary(provider)
                append_jsonl(LEDGER_PATH, {
                    **base, 'status': 'success', **timings,
                    # Includes retries; resume_total_seconds is the successful call itself.
                    'attempt_total_seconds': round(time.perf_counter() - started, 3),
                    'prediction': production_prediction,
                    'raw_prediction': raw_prediction,
                    'production_prediction': production_prediction,
                    'raw_schema_errors': raw_schema_errors(raw_prediction),
                    'raw_schema_valid': not raw_schema_errors(raw_prediction),
                    'reference': example.profile,
                    'field_counts': confusion_rows(production_prediction, example.profile),
                })
                print(
                    f'[{resume_index}/{total}] {example.resume_id}: success in '
                    f"{timings['resume_total_seconds']:.3f}s",
                    flush=True,
                )
                break
            except Exception as exc:
                failure_usage = drain_usage_summary(provider)
                append_jsonl(LEDGER_PATH, {
                    **base, 'status': 'failure',
                    'attempt_total_seconds': round(time.perf_counter() - started, 3),
                    'error_type': type(exc).__name__,
                    'error_message': str(exc)[:500],
                    **failure_usage,
                })
                if attempt == MAX_RETRIES:
                    print(
                        f'[{resume_index}/{total}] {example.resume_id}: failed after '
                        f'{attempt + 1} attempt(s) ({type(exc).__name__})',
                        flush=True,
                    )
                    break
                delay = BASE_RETRY_SECONDS * (2 ** attempt) + random.random()
                print(
                    f'[{resume_index}/{total}] {example.resume_id}: retry '
                    f'{attempt + 2}/{MAX_RETRIES + 1} in {delay:.1f}s',
                    flush=True,
                )
                await asyncio.sleep(delay)

await run_all()
print('Ledger:', LEDGER_PATH)

## 6. Rebuild metrics from saved results

Only the latest successful record per resume for the current model and prompt is included. This makes repeated runs append-only without double-counting.

In [ ]:
ledger = pd.DataFrame(read_ledger())
current = ledger[ledger['experiment_key'].eq(EXPERIMENT_KEY)].copy() if not ledger.empty else ledger
successes = (current[current['status'].eq('success')]
             .sort_values('timestamp_utc')
             .drop_duplicates('resume_id', keep='last'))
failures = current[current['status'].eq('failure')]
print({
    'selected': len(examples), 'successful_unique': len(successes),
    'pending': len(examples) - len(successes), 'logged_failures': len(failures),
})
if len(failures):
    display(failures.groupby('error_type').size().sort_values(ascending=False).rename('attempts').to_frame())

active_references = {example.resume_id: example.profile for example in examples}
count_rows = []
for record in successes.to_dict('records'):
    # Always rescore saved outputs with the evaluator currently visible in
    # this notebook; never mix stale stored metrics into new charts.
    reference = active_references[record['resume_id']]
    prediction = record.get('production_prediction') or record['prediction']
    for row in confusion_rows(prediction, reference):
        count_rows.append({
            'resume_id': record['resume_id'], 'split': record['split'],
            'category': record['category'], **row,
        })
counts = pd.DataFrame(count_rows)
counts.head()

In [ ]:
def safe_div(numerator, denominator):
    return numerator / denominator if denominator else np.nan

def aggregate_metrics(frame: pd.DataFrame, group_cols=('field',)) -> pd.DataFrame:
    frame = frame.copy()
    frame['resume_precision'] = [safe_div(tp, tp + fp) for tp, fp in zip(frame.tp, frame.fp)]
    frame['resume_recall'] = [safe_div(tp, tp + fn) for tp, fn in zip(frame.tp, frame.fn)]
    frame['resume_f1'] = [
        safe_div(2*tp, 2*tp + fp + fn) for tp, fp, fn in zip(frame.tp, frame.fp, frame.fn)
    ]
    sums = (frame.groupby(list(group_cols), as_index=False)[
        ['tp', 'fp', 'fn', 'presence_tp', 'presence_fp', 'presence_fn', 'presence_tn', 'gold_count']
    ].sum())
    sums['precision'] = [safe_div(tp, tp + fp) for tp, fp in zip(sums.tp, sums.fp)]
    sums['recall'] = [safe_div(tp, tp + fn) for tp, fn in zip(sums.tp, sums.fn)]
    sums['f1'] = [
        safe_div(2*tp, 2*tp + fp + fn) for tp, fp, fn in zip(sums.tp, sums.fp, sums.fn)
    ]
    sums['presence_specificity'] = [
        safe_div(tn, tn + fp) for tn, fp in zip(sums.presence_tn, sums.presence_fp)
    ]
    sums['presence_accuracy'] = [
        safe_div(tp + tn, tp + tn + fp + fn)
        for tp, tn, fp, fn in zip(
            sums.presence_tp, sums.presence_tn, sums.presence_fp, sums.presence_fn
        )
    ]
    macro = (frame.groupby(list(group_cols), as_index=False)
             .agg(macro_precision=('resume_precision', 'mean'),
                  macro_recall=('resume_recall', 'mean'),
                  macro_f1=('resume_f1', 'mean'),
                  resume_support=('resume_id', 'nunique')))
    sums = sums.merge(macro, on=list(group_cols), how='left')
    return sums

field_summary = aggregate_metrics(counts).sort_values('f1')
display(field_summary.round(4))

split_summary = aggregate_metrics(counts, ('split', 'field')).sort_values(['split', 'f1'])
category_summary = aggregate_metrics(counts, ('category', 'field'))

field_summary.to_csv(EXPORT_DIR / 'field_summary.csv', index=False)
split_summary.to_csv(EXPORT_DIR / 'field_summary_by_split.csv', index=False)
category_summary.to_csv(EXPORT_DIR / 'field_summary_by_category.csv', index=False)
counts.to_csv(EXPORT_DIR / 'per_resume_field_counts.csv', index=False)

# A searchable evidence index across every model/prompt experiment in the ledger.
all_successes = (ledger[ledger['status'].eq('success')]
                 .sort_values('timestamp_utc')
                 .drop_duplicates(['experiment_key', 'resume_id'], keep='last'))
evidence_rows = []
for record in all_successes.to_dict('records'):
    for field_result in confusion_rows(record['prediction'], record['reference']):
        evidence_rows.append({
            'experiment_key': record['experiment_key'],
            'prompt_version': record.get('prompt_version'),
            'prompt_sha256': record['prompt_sha256'],
            'model': record['model'],
            'resume_id': record['resume_id'],
            'split': record['split'],
            'category': record['category'],
            'field': field_result['field'],
            'tp': field_result['tp'], 'fp': field_result['fp'], 'fn': field_result['fn'],
            'tp_values': field_result.get('tp_values', []),
            'fp_values': field_result.get('fp_values', []),
            'fn_values': field_result.get('fn_values', []),
            'predicted_values': field_result.get('predicted_values', []),
            'gold_values': field_result.get('gold_values', []),
            'resume_total_seconds': record.get('resume_total_seconds'),
            'model_wall_seconds': record.get('model_wall_seconds'),
            'page_calls': record.get('page_calls', []),
        })
evidence = pd.DataFrame(evidence_rows)
evidence.to_json(EXPORT_DIR / 'field_evidence.jsonl', orient='records', lines=True)

def find_evidence(*, resume_id=None, field=None, prompt_version=None, error='any'):
    result = evidence.copy()
    if resume_id is not None:
        result = result[result.resume_id.eq(resume_id)]
    if field is not None:
        result = result[result.field.eq(field)]
    if prompt_version is not None:
        result = result[result.prompt_version.eq(prompt_version)]
    if error == 'fp':
        result = result[result.fp.gt(0)]
    elif error == 'fn':
        result = result[result.fn.gt(0)]
    elif error == 'any':
        result = result[result.fp.gt(0) | result.fn.gt(0)]
    elif error != 'all':
        raise ValueError("error must be 'fp', 'fn', 'any', or 'all'")
    return result.sort_values(['resume_id', 'field', 'prompt_version'])

# Optional report-friendly hierarchy; attempts.jsonl remains the source of truth.
experiment_report = {'generated_at_utc': datetime.now(timezone.utc).isoformat(), 'experiments': []}
for experiment_key, group in all_successes.groupby('experiment_key', sort=False):
    first = group.iloc[0]
    experiment_report['experiments'].append({
        'experiment_key': experiment_key,
        'model': first['model'],
        'prompt_version': first.get('prompt_version'),
        'prompt_sha256': first['prompt_sha256'],
        'system_prompt': first.get('system_prompt'),
        'response_schema': first.get('response_schema'),
        'temperature': first['temperature'],
        'render_dpi': first.get('render_dpi'),
        'resumes': group.to_dict('records'),
    })
(EXPORT_DIR / 'experiment_report.json').write_text(
    json.dumps(
        experiment_report, ensure_ascii=False, indent=2,
        default=lambda value: value.item() if isinstance(value, np.generic) else str(value),
    ), encoding='utf-8'
)

print('Exports written to:', EXPORT_DIR)
display(find_evidence(error='any').head(20))

## 7. Bootstrap confidence intervals and visualizations

Resumes—not individual values—are resampled, preserving the correct unit of analysis. With small support, intervals will correctly remain wide.

In [ ]:
def bootstrap_f1(frame: pd.DataFrame, iterations: int = 2000, seed: int = 42) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    resume_ids = frame['resume_id'].unique()
    samples = {field: [] for field in FIELD_SPECS}
    for _ in range(iterations):
        selected = rng.choice(resume_ids, size=len(resume_ids), replace=True)
        sampled = pd.concat([frame[frame.resume_id.eq(rid)] for rid in selected], ignore_index=True)
        summary = aggregate_metrics(sampled).set_index('field')
        for field in samples:
            samples[field].append(summary.at[field, 'f1'])
    return pd.DataFrame([
        {'field': field, 'f1_ci_low': np.nanpercentile(values, 2.5),
         'f1_ci_high': np.nanpercentile(values, 97.5)}
        for field, values in samples.items()
    ])

confidence = bootstrap_f1(counts)
report = field_summary.merge(confidence, on='field').sort_values('f1')
report.to_csv(EXPORT_DIR / 'field_summary_with_95ci.csv', index=False)

chart_report = report.dropna(subset=['f1', 'f1_ci_low', 'f1_ci_high'])
fig, ax = plt.subplots(figsize=(10, max(6, len(chart_report) * 0.35)))
xerr = np.vstack([chart_report.f1 - chart_report.f1_ci_low,
                  chart_report.f1_ci_high - chart_report.f1])
ax.barh(chart_report.field, chart_report.f1, xerr=xerr, color='#4C78A8', alpha=0.9, capsize=3)
ax.set(xlim=(0, 1), xlabel='Micro F1 (95% bootstrap CI)', ylabel='Field',
       title=f'Vertex vision field extraction — {len(successes)} resumes')
ax.grid(axis='x', alpha=0.25)
plt.tight_layout()
plt.savefig(EXPORT_DIR / 'field_f1_with_95ci.png', dpi=180, bbox_inches='tight')
plt.show()
display(report.round(4))

In [ ]:
# Coverage and error-analysis views. Low support is shown explicitly.
plot_data = report.sort_values('gold_count')
fig, axes = plt.subplots(1, 2, figsize=(15, max(6, len(plot_data) * 0.35)))
axes[0].barh(plot_data.field, plot_data.gold_count, color='#72B7B2')
axes[0].set(title='Gold support by field', xlabel='Annotated values')
axes[1].barh(plot_data.field, plot_data.fp, label='FP', color='#E45756')
axes[1].barh(plot_data.field, plot_data.fn, left=plot_data.fp, label='FN', color='#F2CF5B')
axes[1].set(title='Extraction errors by field', xlabel='Count')
axes[1].legend()
for ax in axes:
    ax.grid(axis='x', alpha=0.25)
plt.tight_layout()
plt.savefig(EXPORT_DIR / 'field_support_and_errors.png', dpi=180, bbox_inches='tight')
plt.show()

## After the baseline run

Review the exact-match results and retained per-resume evidence before defining any follow-up experiment. Keep future experiment results under a distinct experiment identity and ledger.